# Common Libraries

In [14]:
import os, mink, mujoco, mujoco.viewer
from loop_rate_limiters import RateLimiter
import numpy as np

# Model specification

In [3]:
myosim_dir_path = "/home/seojin/myosuite/myosuite/simhive/myo_sim/arm"
os.chdir(myosim_dir_path)

model_name = "myoarm.xml"
xml_string = f"""
        <mujoco model="MyoArm with Mocap">
            <include file="{model_name}"/>
            <worldbody>
                <body name="target" pos="0 0 0" quat="0 1 0 0" mocap="true">
                    <geom type="box" size=".15 .15 .15" contype="0" conaffinity="0" rgba=".6 .3 .3 .2"/>
                </body>
            </worldbody>
        </mujoco>
        """

# Initialize model & data

In [4]:
model = mujoco.MjModel.from_xml_string(xml_string)
data = mujoco.MjData(model)

# IK Configuration

In [5]:
configuration = mink.Configuration(model)

tasks = [
    end_effector_task := mink.FrameTask(
        frame_name="S_grasp",
        frame_type="site",
        position_cost=1.0,
        orientation_cost=1.0,
        lm_damping=1.0,
    ),
    posture_task := mink.PostureTask(model=model, cost=1e-2),
]

solver = "quadprog"
pos_threshold = 1e-4
ori_threshold = 1e-4
max_iters = 20

In [6]:
configuration.update(data.qpos)

In [7]:
posture_task.set_target_from_configuration(configuration)

In [8]:
mujoco.mj_forward(model, data)

In [9]:
mink.move_mocap_to_frame(model, data, "target", "S_grasp", "site")

In [10]:
rate = RateLimiter(frequency=500.0, warn=False) # 시뮬레이션 속도를 일정하게(500Hz) 유지

In [11]:
T_wt = mink.SE3.from_mocap_name(model, data, "target")

In [12]:
end_effector_task.set_target(T_wt)

In [15]:
for i in range(max_iters):
    vel = mink.solve_ik(configuration, tasks, rate.dt, solver, 1e-3) # quadprog 솔버를 사용하여 현재 위치에서 목표 위치로 가기 위한 관절 속도를 계산함.
    configuration.integrate_inplace(vel, rate.dt) # 계산된 속도에 시간 변화량(dt)을 곱해 다음 관절 위치(q)를 업데이트 함.
    err = end_effector_task.compute_error(configuration)
    pos_achieved = np.linalg.norm(err[:3]) <= pos_threshold
    ori_achieved = np.linalg.norm(err[3:]) <= ori_threshold
    if pos_achieved and ori_achieved:
        break

In [16]:
data.qpos[:] = configuration.q # qpos를 실제 MuJoCo 데이터에 적용하고 mj_step을 통해 물리 법칙을 계산함.
mujoco.mj_step(model, data)